# Lesson 3.1 · Pipeline and ColumnTransformer

In module 2 you prepared the data one column type at a time and saw how many silent mistakes come from it: a
`fit` on the test set, columns in the wrong order, an encoder that is not the one from training. The scikit-learn `Pipeline`
solves the problem at the root: the whole path from raw data to prediction becomes **a single object**.

**What you will learn**

- To build a `Pipeline` and understand what happens at every `fit` and every `predict`.
- To handle columns of different types with a `ColumnTransformer`, even with nested pipelines.
- To inspect and modify the parameters of a step without rebuilding anything.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print(X.shape, "| survival rate:", round(y.mean(), 3))
X.head(3)

## The problem: a dataset with everything you need to go wrong

Titanic has numerics with missing values (`age`), categoricals with missing values (`deck`, `port`), a discrete feature (`pclass`) and columns with
different scales (`fare`). To train a model you need to impute, scale, encode, in the right order and with the
statistics of the training set alone. A `Pipeline` lines everything up.

## Pipeline: `fit` and `predict` in a row

```
pipe.fit(X, y):      step1.fit_transform(X, y) → step2.fit_transform(...) → ... → model.fit(...)
pipe.predict(X_new): step1.transform(X_new)    → step2.transform(...)    → ... → model.predict(...)
```

In `fit` every intermediate step learns from the data and passes it to the next; in `predict` the same steps apply **only**
`transform`. It is impossible to refit a scaler on the test set by mistake.

In [ ]:
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
numeric_only = ["pclass", "age", "siblings_spouses", "parents_children", "fare"]
pipe.fit(X_tr[numeric_only], y_tr)
print("test AUC (numerics only):", round(roc_auc_score(y_te, pipe.predict_proba(X_te[numeric_only])[:, 1]), 3))
print("steps:", list(pipe.named_steps))
print("median of age learned from the TRAINING set:", pipe.named_steps["imputer"].statistics_[1])

## ColumnTransformer: one treatment per group of columns

Every row of the list is `(name, transformer, columns)`. Columns are given by name, by position or with a *selector*
(`make_column_selector(dtype_include=...)`). Every transformer can itself be a `Pipeline`: here every group has
imputation first and then the transformation.

In [ ]:
numeric = ["age", "siblings_spouses", "parents_children", "fare"]
categorical = ["sex", "port", "deck", "title"]

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), categorical),
    ("pclass", "passthrough", ["pclass"]),                    # already 1, 2, 3: we leave it as it is
], sparse_threshold=0)

model = make_pipeline(prep, LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
print("test AUC:", round(roc_auc_score(y_te, model.predict_proba(X_te)[:, 1]), 3))

> **Warning.** The columns **not listed** in a `ColumnTransformer` are silently dropped (`remainder="drop"`, the default).
> To keep them use `remainder="passthrough"`. It is one of the bugs you will find in exercise E2.

## Inspecting and modifying

A pipeline is an inspectable object: you access the steps by name, take a piece with slicing and change the
parameters with the `step__parameter` syntax (which is the same one used by `GridSearchCV` in lesson 3.3).

In [ ]:
print("feature names after the preprocessing:", len(model[:-1].get_feature_names_out()))
print(list(model[:-1].get_feature_names_out())[:6], "...")

model.set_params(logisticregression__C=0.1)                   # change a parameter of a nested step
model.fit(X_tr, y_tr)
coef = pd.Series(model[-1].coef_[0], index=model[:-1].get_feature_names_out())
coef.abs().nlargest(4).round(2)

## Columns by type, without listing them

`make_column_selector` chooses the columns by data type: useful when the schema changes, less so when you want explicit
control (in production an explicit list is often preferable, because a badly inferred type changes the model silently).

In [ ]:
prep2 = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include="number")),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")),
     make_column_selector(dtype_include=object)),
], sparse_threshold=0)
p2 = make_pipeline(prep2, LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
print("test AUC with the selectors:", round(roc_auc_score(y_te, p2.predict_proba(X_te)[:, 1]), 3))

> **Common mistake.** Fitting the scaler or the imputer on the whole dataset **before** splitting or doing the cross-validation: the statistics of the validation fold enter the training. Inside a `Pipeline` it is impossible to do it; outside it is very easy (exercises E1 and E3 of module 1), and a `ColumnTransformer` that silently drops the columns not listed is the twin mistake (exercise E2).

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>Why does the pipeline avoid preprocessing leakage?</summary>

Because `fit` calls `fit_transform` on the intermediate steps **only on the training data it receives**, and `predict` calls only `transform`. Inside `cross_val_score` or `GridSearchCV` the pipeline is cloned and refitted at every fold on the training part of the fold alone: the statistics of the validation fold never enter the preprocessing.

</details>

<details><summary>When is `make_column_selector` convenient and when the explicit list?</summary>

The selector is handy in exploration. In production an explicit list is safer: if a numeric column arrives by mistake as text, a selector silently moves it to the group of categoricals and the model changes without warning.

</details>